In [1]:
import os
import gc
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from torch.amp import autocast, GradScaler
import torchvision.models as tvm
import albumentations as Albu
import optuna
from optuna.samplers import TPESampler
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, f1_score,
    classification_report, confusion_matrix,
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from IPython.display import display
import sys
sys.path.append('..')
from utils.dataset import PandasOverlapDataset

optuna.logging.set_verbosity(optuna.logging.WARNING)

/home/prof_antoniooseas/woshington/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
REPO_DIR   = '..'                                   # pos-propose/family -> repo root
DATA_DIR   = os.path.join(REPO_DIR, 'data')
IMAGES_DIR = os.path.join(REPO_DIR, 'dataset')

LOG_DIR    = 'logs'
MODEL_DIR  = 'models'
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

In [ ]:
df_all = pd.read_csv(os.path.join(DATA_DIR, 'train_5fold.csv'))
df_all.columns = df_all.columns.str.strip()
print(f'Total records: {len(df_all)}')

# --- Noise cleaning: drop the noisiest images by difficulty score ---
df_entropy = pd.read_csv(os.path.join(DATA_DIR, 'entropy.csv'))
df_entropy = df_entropy.sort_values('difficulty_score', ascending=False)
n_remove   = int(len(df_entropy) * ENTROPY_DROP_FRAC)
noisy_ids  = set(df_entropy.head(n_remove)['image_id'])
df_all     = df_all[~df_all['image_id'].isin(noisy_ids)].reset_index(drop=True)
print(f'After entropy filter (removed {n_remove} noisiest): {len(df_all)}')


def drop_missing(df):
    exists = df['image_id'].apply(lambda x: os.path.isdir(os.path.join(IMAGES_DIR, str(x))))
    return df[exists].reset_index(drop=True)


df_train = drop_missing(df_all[df_all['fold'] != 3].reset_index(drop=True))
df_val   = drop_missing(df_all[df_all['fold'] == 3].reset_index(drop=True))

print(f'Train: {len(df_train)}   Val: {len(df_val)}')
print('Val class distribution:')
print(df_val['isup_grade'].value_counts().sort_index())

train_ds = PandasOverlapDataset(IMAGES_DIR, df_train, transforms=None, overlap=0)